In [1]:
# Cell 1: 基础环境 & 随机种子

import os
import random
import numpy as np
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.data import HeteroData

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, accuracy_score, recall_score

import pandas as pd

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)

def set_seed(seed=51):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(51)


Device: cuda


In [2]:
# Cell 2: 加载 hetero_data.pt，并准备元信息

HET_PATH = os.path.join(".", "data", "hetero_data.pt")
data_obj = torch.load(HET_PATH)

if isinstance(data_obj, HeteroData):
    hetero_data = data_obj
elif isinstance(data_obj, dict) and 'hetero_data' in data_obj:
    hetero_data = data_obj['hetero_data']
else:
    raise ValueError("无法识别 hetero_data 格式，请确认 hetero_data.pt 里存的是 HeteroData 或 {'hetero_data': HeteroData}.")

print(hetero_data)

# 元信息
node_types, edge_types = hetero_data.metadata()
print("Node types:", node_types)
print("Edge types:", edge_types)

# 每种节点类型的输入维度
in_dims = {ntype: hetero_data[ntype].x.size(1) for ntype in node_types}
print("Input dims per node type:", in_dims)

# 拷贝一份特征到 CPU（训练时再整体搬到 device）
x_dict = {ntype: hetero_data[ntype].x.clone() for ntype in node_types}


HeteroData(
  protein={
    x=[18326, 150],
    y=[18326],
  },
  lncrna={ x=[1811, 150] },
  mirna={ x=[1589, 150] },
  (lncrna, lncrnaTOlncrna, lncrna)={ edge_index=[2, 296] },
  (lncrna, lncrnaTOmirna, mirna)={ edge_index=[2, 7277] },
  (lncrna, lncrnaTOprotein, protein)={ edge_index=[2, 10376] },
  (mirna, mirnaTOlncrna, lncrna)={ edge_index=[2, 7277] },
  (mirna, mirnaTOmirna, mirna)={ edge_index=[2, 158] },
  (mirna, mirnaTOprotein, protein)={ edge_index=[2, 510229] },
  (protein, proteinTOlncrna, lncrna)={ edge_index=[2, 10376] },
  (protein, proteinTOmirna, mirna)={ edge_index=[2, 510229] },
  (protein, proteinTOprotein, protein)={ edge_index=[2, 939078] }
)
Node types: ['protein', 'lncrna', 'mirna']
Edge types: [('lncrna', 'lncrnaTOlncrna', 'lncrna'), ('lncrna', 'lncrnaTOmirna', 'mirna'), ('lncrna', 'lncrnaTOprotein', 'protein'), ('mirna', 'mirnaTOlncrna', 'lncrna'), ('mirna', 'mirnaTOmirna', 'mirna'), ('mirna', 'mirnaTOprotein', 'protein'), ('protein', 'proteinTOlncrna', 'lnc

In [3]:
# 假设 metadata = (node_types, edge_types)
_, edge_types = hetero_data.metadata()
missing = [e for e in edge_types if e not in hetero_data.edge_index_dict]
extra   = [e for e in hetero_data.edge_index_dict.keys() if e not in edge_types]
print("missing edge_types:", missing)
print("extra edge_types:", extra)
assert len(missing) == 0, "Some edge types in metadata are missing in edge_index_dict!"


missing edge_types: []
extra edge_types: []


In [4]:
# Cell 2.1: 构造不同消融实验的 metadata

full_node_types, full_edge_types = hetero_data.metadata()
print("Full node types :", full_node_types)
print("Full edge types :", full_edge_types)

def make_metadata(mode: str):
    """
    mode:
        'full'         : 原始模型，保留所有关系
        'no_lnc'       : 去除所有与 lncrna 相关的边
        'no_mir'       : 去除所有与 mirna 相关的边
        'no_lnc_mir'   : 去除 lncrna & mirna，只保留 PPI (protein-protein)
    """
    if mode == 'full':
        edge_types = full_edge_types

    elif mode == 'no_lnc':
        edge_types = [
            et for et in full_edge_types
            if (et[0] != 'lncrna' and et[2] != 'lncrna')
        ]

    elif mode == 'no_mir':
        edge_types = [
            et for et in full_edge_types
            if (et[0] != 'mirna' and et[2] != 'mirna')
        ]

    elif mode == 'no_lnc_mir':
        edge_types = [
            et for et in full_edge_types
            if (et[0] == 'protein' and et[2] == 'protein')
        ]
    else:
        raise ValueError(f"Unknown ablation mode: {mode}")

    print(f"[{mode}] edge_types used:")
    for et in edge_types:
        print("   ", et)

    return (full_node_types, edge_types)


Full node types : ['protein', 'lncrna', 'mirna']
Full edge types : [('lncrna', 'lncrnaTOlncrna', 'lncrna'), ('lncrna', 'lncrnaTOmirna', 'mirna'), ('lncrna', 'lncrnaTOprotein', 'protein'), ('mirna', 'mirnaTOlncrna', 'lncrna'), ('mirna', 'mirnaTOmirna', 'mirna'), ('mirna', 'mirnaTOprotein', 'protein'), ('protein', 'proteinTOlncrna', 'lncrna'), ('protein', 'proteinTOmirna', 'mirna'), ('protein', 'proteinTOprotein', 'protein')]


In [5]:
# Cell 2.2: 随机遮掩部分边（按比例删除），用于鲁棒性消融

def random_mask_edges(hetero_data, drop_ratio: float, seed: int = 51):
    """
    在 hetero_data 上随机删除 drop_ratio 比例的边（每种关系分别按比例删），
    不改动节点和标签，返回一个新的 HeteroData。
    drop_ratio = 0.1 表示删除 10% 的边。
    """
    assert 0.0 <= drop_ratio < 1.0
    if drop_ratio == 0.0:
        return hetero_data  # 不删边，直接返回原图

    data = hetero_data.clone()
    g = torch.Generator()
    g.manual_seed(seed)

    for edge_type in data.edge_types:
        # edge_type 形如 ('lncrna','lnc-protein','protein')
        edge_index = data[edge_type].edge_index
        num_edges = edge_index.size(1)

        keep_num = int(num_edges * (1.0 - drop_ratio))
        if keep_num <= 0:
            print(f"[WARN] edge_type {edge_type} all edges would be dropped, skip.")
            continue

        perm = torch.randperm(num_edges, generator=g)
        keep_idx = perm[:keep_num]

        data[edge_type].edge_index = edge_index[:, keep_idx]

        # 如果你有 edge_weight / edge_attr 的话，一起裁剪
        if 'edge_weight' in data[edge_type]:
            data[edge_type].edge_weight = data[edge_type].edge_weight[keep_idx]
        if 'edge_attr' in data[edge_type]:
            data[edge_type].edge_attr = data[edge_type].edge_attr[keep_idx]

        print(f"[drop {drop_ratio:.2f}] {edge_type}: {num_edges} -> {keep_num} edges")

    return data


In [6]:
# （加入了注意力缓存）Cell 3: 定义 MRAGDM 模型（多路 + 注意力 + 扩散）与 ProteinClassifier
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing


class CouplingAttention(nn.Module):
    """
    多路耦合注意力：在同一节点类型上融合不同关系得到的表示。
    输入: x: [N, R, D]
    输出: out: [N, D]
    额外：可选缓存 last_weight: [N, R]
    """
    def __init__(self, in_dim: int, dim_a: int):
        super().__init__()
        self.proj = nn.Linear(in_dim, dim_a)
        self.attn = nn.Linear(dim_a, 1)
        self.norm = nn.LayerNorm(in_dim)
        self.residual = nn.Sequential(
            nn.Linear(in_dim, in_dim * 2),
            nn.GELU(),
            nn.Linear(in_dim * 2, in_dim)
        )
        self.dropout = nn.Dropout(0.3)

        # ====== NEW: 注意力缓存开关 ======
        self.cache_attn = False
        self.last_weight = None  # [N, R]

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [N, R, D]
        h = torch.tanh(self.proj(x))         # [N, R, dim_a]
        score = self.attn(h).squeeze(-1)     # [N, R]
        weight = F.softmax(score, dim=-1)    # [N, R]

        # ====== NEW: 缓存 ======
        if self.cache_attn:
            self.last_weight = weight.detach()

        out = torch.sum(x * weight.unsqueeze(-1), dim=1)  # [N, D]
        out = self.norm(out + self.residual(out))
        return self.dropout(out)


class DiffusionRelConv(MessagePassing):
    """
    MRAGDM 的核心：在关系感知的注意力消息传递上加入扩散模块。
    扩散过程 (APPNP 风格):
        h^{k+1} = (1 - alpha) * A_attn h^{k} + alpha * h^0
    其中 A_attn 由基于 (src, dst) 的注意力给出。
    额外：可选缓存 last_alpha: [E, 1] 和 last_edge_index: [2, E]
    """
    def __init__(self,
                 in_dim: int,
                 out_dim: int,
                 K: int = 3,
                 alpha: float = 0.1,
                 dropout: float = 0.5,
                 activation: nn.Module = nn.ReLU()):
        super().__init__(aggr='add')  # 对 dst 聚合
        self.lin_src = nn.Linear(in_dim, out_dim)
        self.lin_dst = nn.Linear(in_dim, out_dim)

        self.attn_proj = nn.Linear(out_dim, out_dim)
        self.score_fn = nn.Parameter(torch.Tensor(out_dim))
        nn.init.xavier_uniform_(self.score_fn.unsqueeze(0))

        self.K = K
        self.alpha = alpha
        self.dropout = nn.Dropout(dropout)
        self.activation = activation
        self.layer_norm = nn.LayerNorm(out_dim)

        # ====== NEW: 注意力缓存开关 ======
        self.cache_attn = False
        self.last_alpha = None       # [E, 1]
        self.last_edge_index = None  # [2, E]

    def forward(self, x, edge_index, size=None):
        """
        x: (x_src, x_dst) 或 [N, F]
        edge_index: [2, E]
        size: (num_src, num_dst)
        """
        if isinstance(x, tuple):
            x_src, x_dst = x
        else:
            x_src = x_dst = x

        h_src0 = self.lin_src(x_src)   # 源节点初始
        h_dst0 = self.lin_dst(x_dst)   # 目标节点初始

        num_src = h_src0.size(0)
        num_dst = h_dst0.size(0)
        if size is None:
            size = (num_src, num_dst)

        # ====== NEW: 缓存 edge_index（用于对齐 last_alpha） ======
        if self.cache_attn:
            self.last_edge_index = edge_index.detach()
            self.last_alpha = None  # 每次 forward 前清空，避免读到旧的

        h = h_dst0
        for _ in range(self.K):
            # 注意：这里传入 x=(h_src0, h)，在 message 中用 x_j / x_i 拆分
            h = self.propagate(edge_index, x=(h_src0, h), size=size)
            h = (1.0 - self.alpha) * h + self.alpha * h_dst0

        h = self.dropout(self.activation(h))
        h = self.layer_norm(h + h_dst0)
        return h

    def message(self, x_j, x_i):
        """
        x_j: [E, D] 源节点（邻居）在每条边上的表示
        x_i: [E, D] 目标节点（中心）在每条边上的当前表示
        """
        attn_input = torch.tanh(self.attn_proj(x_j + x_i))     # [E, D]
        score = (attn_input * self.score_fn).sum(dim=-1, keepdim=True)  # [E, 1]
        alpha = torch.sigmoid(score)                           # [E, 1]

        # ====== NEW: 缓存边注意力 ======
        if self.cache_attn:
            # 注意：propagate 内部可能多次调用 message；这里保留“最后一次传播”的 alpha
            self.last_alpha = alpha.detach()

        return alpha * x_j                                     # [E, D]

    def update(self, aggr_out):
        # aggr_out: [N_dst, D]
        return aggr_out


class MRAGDMLayer(nn.Module):
    """
    单层 MRAGDM：
    - 每条 (src, rel, dst) 关系一个 DiffusionRelConv（多路）
    - 对同一 dst 类型的多路结果用 CouplingAttention 融合（注意力）
    - 再加残差 + LayerNorm
    """
    def __init__(self,
                 metadata,
                 hidden_dim: int,
                 dim_a: int,
                 use_diffusion: bool = True,
                 diffusion_steps: int = 3,
                 diffusion_alpha: float = 0.1,
                 dropout: float = 0.5,
                 activation: str = 'relu'):
        super().__init__()
        node_types, edge_types = metadata
        self.edge_types = edge_types
        self.hidden_dim = hidden_dim
        self.node_types = node_types

        if activation is None:
            act = nn.ReLU()
        else:
            act_cls = getattr(nn, activation.upper(), nn.ReLU)
            act = act_cls()

        # 扩散参数
        if use_diffusion:
            K = diffusion_steps
            alpha = diffusion_alpha
        else:
            K = 1
            alpha = 0.0

        # 每条关系一个扩散卷积
        self.rel_convs = nn.ModuleDict()
        for (src, rel, dst) in edge_types:
            key = f"{src}__{rel}__{dst}"
            self.rel_convs[key] = DiffusionRelConv(
                in_dim=hidden_dim,
                out_dim=hidden_dim,
                K=K,
                alpha=alpha,
                dropout=dropout,
                activation=act
            )

        # 关系级注意力（多路融合）
        self.coupling_attention = CouplingAttention(
            in_dim=hidden_dim,
            dim_a=dim_a
        )

        # 每个节点类型的残差映射 & 归一化
        self.skip_proj = nn.ModuleDict({
            ntype: nn.Linear(hidden_dim, hidden_dim) for ntype in node_types
        })
        self.final_norm = nn.ModuleDict({
            ntype: nn.LayerNorm(hidden_dim) for ntype in node_types
        })
        self.res_alpha = nn.Parameter(torch.tensor(0.5))

    def forward(self, h_dict, edge_index_dict):
        """
        h_dict: {ntype: [N_ntype, hidden_dim]}
        edge_index_dict: {(src, rel, dst): edge_index}
        """
        all_h_rel = {ntype: [] for ntype in h_dict.keys()}

        for (src, rel, dst) in self.edge_types:
            key = f"{src}__{rel}__{dst}"
            conv = self.rel_convs[key]
            edge_index = edge_index_dict[(src, rel, dst)]

            # 确定双向图的 (num_src, num_dst)
            num_src = h_dict[src].size(0)
            num_dst = h_dict[dst].size(0)

            h_dst_rel = conv(
                (h_dict[src], h_dict[dst]),
                edge_index,
                size=(num_src, num_dst)
            )  # [N_dst, D]

            all_h_rel[dst].append(h_dst_rel)

        out_dict = {}
        alpha = torch.sigmoid(self.res_alpha)

        for ntype, h in h_dict.items():
            if len(all_h_rel[ntype]) == 0:
                # 没有作为 dst 出现过，维持自身
                h_prev = self.skip_proj[ntype](h)
                out = self.final_norm[ntype](h_prev)
            else:
                # [N, R_ntype, D]
                h_stack = torch.stack(all_h_rel[ntype], dim=1)
                h_new = self.coupling_attention(h_stack)  # [N, D]
                h_prev = self.skip_proj[ntype](h)
                h_comb = alpha * h_new + (1.0 - alpha) * h_prev
                out = self.final_norm[ntype](h_comb)
            out_dict[ntype] = out

        return out_dict


class MRAGDM(nn.Module):
    """
    MRAGDN: Multi-Relation Attention Graph Diffusion Network
    - 先将各节点类型映射到统一 hidden_dim
    - 堆叠多层 MRAGDMLayer（多路 + 注意力 + 扩散）
    - 输出每种节点类型的隐藏表示 h_dict
    """
    def __init__(self,
                 metadata,
                 in_dims: dict,
                 hidden_dim: int,
                 out_dim: int,
                 num_layers: int,
                 node_types,
                 use_diffusion: bool = True,
                 diffusion_steps: int = 3,
                 diffusion_alpha: float = 0.1):
        super().__init__()
        self.metadata = metadata
        self.node_types = node_types
        self.hidden_dim = hidden_dim

        # 不同节点类型输入维度可能不同，先映射到统一 hidden_dim
        self.input_proj = nn.ModuleDict({
            ntype: nn.Linear(in_dims[ntype], hidden_dim) for ntype in node_types
        })

        # 堆叠多层 MRAGDMLayer
        self.layers = nn.ModuleList()
        for _ in range(num_layers):
            self.layers.append(
                MRAGDMLayer(
                    metadata=metadata,
                    hidden_dim=hidden_dim,
                    dim_a=64,
                    use_diffusion=use_diffusion,
                    diffusion_steps=diffusion_steps,
                    diffusion_alpha=diffusion_alpha,
                    dropout=0.5,
                    activation='relu'
                )
            )

        # 可选的输出投影
        self.out_proj = nn.ModuleDict({
            ntype: nn.Linear(hidden_dim, out_dim) for ntype in node_types
        })

    def forward(self, x_dict, hetero_data):
        # 1. 将各节点类型特征映射到 hidden_dim
        h_dict = {
            ntype: self.input_proj[ntype](x_dict[ntype])
            for ntype in self.node_types
        }

        # 2. 多层 MRAGDMLayer
        for layer in self.layers:
            h_dict = layer(h_dict, hetero_data.edge_index_dict)

        # 3. 这里返回 hidden_dim 的表示；out_proj 可按需使用
        return h_dict


class NodeClassifier(nn.Module):
    def __init__(self, embed_dim: int, num_classes: int, dropout: float = 0.1):
        super().__init__()
        self.fc1 = nn.Linear(embed_dim, 128)
        self.fc2 = nn.Linear(128, num_classes)
        self.dropout = nn.Dropout(dropout)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x


class ProteinClassifier(nn.Module):
    """
    包裹 MRAGDM 编码器，专门做 protein 节点的二分类。
    forward 输出: (logits, h_dict)
    """
    def __init__(self, encoder: MRAGDM, hidden_dim: int, num_classes: int, target_ntype: str = 'protein'):
        super().__init__()
        self.encoder = encoder
        self.target_ntype = target_ntype
        self.classifier = NodeClassifier(hidden_dim, num_classes, dropout=0.1)

    def forward(self, x_dict, hetero_data):
        h_dict = self.encoder(x_dict, hetero_data)
        h_protein = h_dict[self.target_ntype]  # [N_protein, hidden_dim]
        logits = self.classifier(h_protein)    # [N_protein, num_classes]
        return logits, h_dict


In [8]:
# Cell 4: 使用 MRAGDM + ProteinClassifier 做嵌套交叉验证训练
# 一次性跑：4种结构消融 + 5个随机删边（只对full）
# 输出：./  下每个实验单独文件夹，内部保存文件命名与原始一致

import os, random
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, accuracy_score, recall_score

# ===============================
# 0) 固定 seed
# ===============================
SEED = 51
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# 可复现（会稍慢）
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ===============================
# 1) 检查标签
# ===============================
if 'y' not in hetero_data['protein']:
    raise ValueError("protein 节点缺少标签 y，请在 hetero_data['protein'].y 中提供标签（0/1，未标注可为-1）。")

protein_y = hetero_data['protein'].y.view(-1)          # CPU
labeled_mask = protein_y >= 0
labeled_idx = labeled_mask.nonzero(as_tuple=False).view(-1)
labeled_y = protein_y[labeled_idx].cpu().numpy()

print(f"Total protein nodes     : {hetero_data['protein'].num_nodes}")
print(f"Labeled protein nodes   : {labeled_idx.numel()}")

# ===============================
# 2) 特征放到 device（图会在每个实验里单独构造后再 .to(device)）
# ===============================
x_dict_device = {k: v.to(device) for k, v in x_dict.items()}

# ===============================
# 3) 外层：固定一次 Train/Test = 75% / 25%（用 SEED 控制）
# ===============================
outer_test_size = 0.25
train_all_idx_np, test_idx_np, train_all_y, test_y = train_test_split(
    labeled_idx.cpu().numpy(),
    labeled_y,
    test_size=outer_test_size,
    stratify=labeled_y,
    random_state=SEED
)
print(f"Outer train size (labeled): {len(train_all_idx_np)}")
print(f"Outer test  size (labeled): {len(test_idx_np)}")

# ===============================
# 4) 内层：10 折（用 SEED 控制）
# ===============================
n_splits = 10
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)

# 早停
patience = 30
max_epochs = 300

# ===============================
# 5) 定义本次要跑的“实验列表”
#    - 4个结构消融：full/no_lnc/no_mir/no_lnc_mir（不删边）
#    - 5个删边：只对full做 drop_ratio=0.1~0.5
# ===============================
ABLATION_MODES = ['full' ]
DROP_RATIOS = [0.1, 0.2, 0.3, 0.4, 0.5]
DIFFUSION_STEPS_LIST = [1, 3]
experiments = []

for K in DIFFUSION_STEPS_LIST:
    experiments.append({
        "name": f"full_K{K}",
        "mode": "full",
        "drop_ratio": 0.0,
        "K": K
    })

# ===============================
# 6) 根目录：当前目录下   文件夹
# ===============================
root_dir = f"./cv_results_diffusion_K "
os.makedirs(root_dir, exist_ok=True)
print("All results will be saved under:", root_dir)

# ===============================
# 7) 逐个实验跑（每个实验一个文件夹，内部保存与你原始一致）
# ===============================
for exp in experiments:
    EXP_NAME = exp["name"]
    ABLATION_MODE = exp["mode"]
    EDGE_DROP_RATIO = exp["drop_ratio"]
    K_STEPS = exp["K"]
    print("\n" + "="*120)
    print(f"K: {K_STEPS} |Experiment: {EXP_NAME} | mode={ABLATION_MODE} | drop_ratio={EDGE_DROP_RATIO}")
    print("="*120)

    # 7.1 构造 metadata（决定保留哪些 edge_types）
    metadata = make_metadata(ABLATION_MODE)
    node_types, edge_types = metadata[0], metadata[1]

    # 7.2 构造该实验用的图（先按 mode 裁剪边类型，再可选随机删边）
    hetero_data_used = hetero_data.edge_type_subgraph(edge_types)
    if EDGE_DROP_RATIO > 0:
        hetero_data_used = random_mask_edges(hetero_data_used, drop_ratio=EDGE_DROP_RATIO, seed=SEED)

    hetero_data_device = hetero_data_used.to(device)

    # 7.3 保存目录： /EXP_NAME
    save_dir = os.path.join(root_dir, EXP_NAME)
    os.makedirs(save_dir, exist_ok=True)
    print("Results will be saved to:", save_dir)

    outer_test_results = []

    # 7.4 内层10折训练
    for fold, (train_sub_idx, val_sub_idx) in enumerate(skf.split(train_all_idx_np, train_all_y)):
        print(f"\n========== [{EXP_NAME}] Inner Fold {fold+1}/{n_splits} ==========")

        train_nodes = torch.from_numpy(train_all_idx_np[train_sub_idx])  # CPU
        val_nodes   = torch.from_numpy(train_all_idx_np[val_sub_idx])    # CPU

        # 初始化模型
        encoder = MRAGDM(
            metadata=metadata,
            in_dims=in_dims,
            hidden_dim=64,
            out_dim=48,
            num_layers=2,
            node_types=node_types,
            use_diffusion=True,
            diffusion_steps=K_STEPS,
            diffusion_alpha=0.2
        ).to(device)

        model = ProteinClassifier(encoder, hidden_dim=64, num_classes=2, target_ntype='protein').to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=5e-5)

        history = []
        best_auprc = 0.0
        best_state_dict = None
        best_val_metrics = None
        best_epoch = 0

        for epoch in range(1, max_epochs + 1):
            # ===== Train =====
            model.train()
            optimizer.zero_grad()

            logits, _ = model(x_dict_device, hetero_data_device)

            train_logits = logits[train_nodes.to(device)]
            train_labels = protein_y[train_nodes].to(device).long()

            loss = F.cross_entropy(train_logits, train_labels)
            loss.backward()
            optimizer.step()

            # ===== Val =====
            model.eval()
            with torch.no_grad():
                logits_eval, _ = model(x_dict_device, hetero_data_device)

                val_logits = logits_eval[val_nodes.to(device)]
                val_prob = F.softmax(val_logits, dim=-1)[:, 1].cpu().numpy()
                val_true = protein_y[val_nodes].cpu().numpy().astype(int)
                val_pred = (val_prob >= 0.5).astype(int)

                try:
                    val_auroc = roc_auc_score(val_true, val_prob)
                except ValueError:
                    val_auroc = 0.0
                val_auprc = average_precision_score(val_true, val_prob)
                val_f1 = f1_score(val_true, val_pred)
                val_acc = accuracy_score(val_true, val_pred)
                val_recall = recall_score(val_true, val_pred, zero_division=0)

            history.append({
                'epoch': epoch,
                'train_loss': float(loss.item()),
                'val_acc': float(val_acc),
                'val_auroc': float(val_auroc),
                'val_auprc': float(val_auprc),
                'val_f1': float(val_f1),
                'val_recall': float(val_recall),
            })

            # 早停：按 val AUPRC
            if val_auprc > best_auprc + 1e-6:
                best_auprc = val_auprc
                best_state_dict = {k: v.cpu() for k, v in model.state_dict().items()}
                best_val_metrics = (val_acc, val_auroc, val_auprc, val_f1, val_recall)
                best_epoch = epoch
            else:
                if epoch - best_epoch >= patience:
                    print(f"Early stopping at epoch {epoch}, no AUPRC improvement for {patience} epochs.")
                    break

            if epoch % 10 == 0 or epoch == 1:
                print(f"Epoch {epoch:03d} | Loss {loss.item():.4f} | "
                      f"Val AUPRC {val_auprc:.4f} | Val AUROC {val_auroc:.4f} | "
                      f"Val F1 {val_f1:.4f} | Val ACC {val_acc:.4f} | Val Recall {val_recall:.4f}")

        # ===== Test with best model =====
        if best_state_dict is None:
            print(f"[{EXP_NAME} | Fold {fold}] WARNING: no best_state_dict, please check training.")
            continue

        print(f"[{EXP_NAME} | Fold {fold}] Best Val AUPRC={best_auprc:.4f} at epoch {best_epoch}")

        encoder_best = MRAGDM(
            metadata=metadata,
            in_dims=in_dims,
            hidden_dim=64,
            out_dim=48,
            num_layers=2,
            node_types=node_types,
            use_diffusion=True,
            diffusion_steps=K,
            diffusion_alpha=0.2
        ).to(device)

        model_best = ProteinClassifier(encoder_best, hidden_dim=64, num_classes=2, target_ntype='protein').to(device)
        model_best.load_state_dict({k: v.to(device) for k, v in best_state_dict.items()})
        model_best.eval()

        with torch.no_grad():
            logits_all, _ = model_best(x_dict_device, hetero_data_device)

            prob_all_t = F.softmax(logits_all, dim=-1)[:, 1]
            prob_all = prob_all_t.cpu().numpy()

            # 仅画图缩放（不影响评估）
            gamma = 0.6
            p = prob_all_t.clamp(0.0, 1.0)
            p_scaled = p.clone()
            left_mask = (p <= 0.5)
            right_mask = ~left_mask
            p_scaled[left_mask] = 0.5 * ((p[left_mask] / 0.5).pow(gamma))
            p_scaled[right_mask] = 1.0 - 0.5 * (((1.0 - p[right_mask]) / 0.5).pow(gamma))
            prob_all_scaled = p_scaled.cpu().numpy()

            # 外层固定测试集评估（仍用 prob_all）
            test_prob = prob_all[test_idx_np]
            test_true = test_y.astype(int)
            test_pred = (test_prob >= 0.5).astype(int)

            try:
                test_auroc = roc_auc_score(test_true, test_prob)
            except ValueError:
                test_auroc = 0.0
            test_auprc = average_precision_score(test_true, test_prob)
            test_f1 = f1_score(test_true, test_pred)
            test_acc = accuracy_score(test_true, test_pred)
            test_recall = recall_score(test_true, test_pred, zero_division=0)

        print(f"[{EXP_NAME} | Fold {fold}] Test ACC={test_acc:.4f} | AUROC={test_auroc:.4f} | "
              f"AUPRC={test_auprc:.4f} | F1={test_f1:.4f} | Recall={test_recall:.4f}")

        outer_test_results.append({
            'fold': fold,
            'val_acc': best_val_metrics[0],
            'val_auroc': best_val_metrics[1],
            'val_auprc': best_val_metrics[2],
            'val_f1': best_val_metrics[3],
            'val_recall': best_val_metrics[4],
            'test_acc': test_acc,
            'test_auroc': test_auroc,
            'test_auprc': test_auprc,
            'test_f1': test_f1,
            'test_recall': test_recall,
        })

        # ===== 保存：保持原始文件命名 =====
        model_path = os.path.join(save_dir, f"best_model_fold{fold}_testAUPRC{test_auprc:.4f}.pt")
        torch.save(best_state_dict, model_path)

        df_pred = pd.DataFrame({
            'protein_node_id': np.arange(hetero_data['protein'].num_nodes),
            'prob_cancer': prob_all_scaled,
            'label': protein_y.cpu().numpy()
        })
        df_pred.to_csv(os.path.join(save_dir, f"best_fold{fold}_all_protein_predictions.csv"), index=False)

        pd.DataFrame(history).to_csv(os.path.join(save_dir, f"train_history_fold{fold}.csv"), index=False)

    # ===== 汇总：outer_test_results_summary.csv（保持原始命名）=====
    print(f"\n========== Summary [{EXP_NAME}] (on fixed 25% test set) ==========")
    if len(outer_test_results) > 0:
        df_res = pd.DataFrame(outer_test_results)

        mean_row = {
            'fold': 'Mean',
            'val_acc': df_res['val_acc'].mean(),
            'val_auroc': df_res['val_auroc'].mean(),
            'val_auprc': df_res['val_auprc'].mean(),
            'val_f1': df_res['val_f1'].mean(),
            'val_recall': df_res['val_recall'].mean(),
            'test_acc': df_res['test_acc'].mean(),
            'test_auroc': df_res['test_auroc'].mean(),
            'test_auprc': df_res['test_auprc'].mean(),
            'test_f1': df_res['test_f1'].mean(),
            'test_recall': df_res['test_recall'].mean(),
        }
        df_res = pd.concat([df_res, pd.DataFrame([mean_row])], ignore_index=True)
        df_res_path = os.path.join(save_dir, "outer_test_results_summary.csv")
        df_res.to_csv(df_res_path, index=False)

        print("Per-fold results saved to:", df_res_path)
        print("Mean Test AUPRC :", mean_row['test_auprc'])
    else:
        print("No valid fold results, 请检查标签/训练过程。")

Total protein nodes     : 18326
Labeled protein nodes   : 2856
Outer train size (labeled): 2142
Outer test  size (labeled): 714
All results will be saved under: ./cv_results_diffusion_K 

K: 1 |Experiment: full_K1 | mode=full | drop_ratio=0.0
[full] edge_types used:
    ('lncrna', 'lncrnaTOlncrna', 'lncrna')
    ('lncrna', 'lncrnaTOmirna', 'mirna')
    ('lncrna', 'lncrnaTOprotein', 'protein')
    ('mirna', 'mirnaTOlncrna', 'lncrna')
    ('mirna', 'mirnaTOmirna', 'mirna')
    ('mirna', 'mirnaTOprotein', 'protein')
    ('protein', 'proteinTOlncrna', 'lncrna')
    ('protein', 'proteinTOmirna', 'mirna')
    ('protein', 'proteinTOprotein', 'protein')
Results will be saved to: ./cv_results_diffusion_K /full_K1

========== [full_K1] Inner Fold 1/10 ==========
Epoch 001 | Loss 0.6091 | Val AUPRC 0.4278 | Val AUROC 0.6346 | Val F1 0.0000 | Val ACC 0.7209 | Val Recall 0.0000
Epoch 010 | Loss 0.4774 | Val AUPRC 0.5322 | Val AUROC 0.7525 | Val F1 0.5479 | Val ACC 0.6930 | Val Recall 0.6667
Epoch 0